<a href="https://colab.research.google.com/github/ashandish/FakeNewsDetection/blob/main/Fakeddit_Multimodal_Classifier_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import shutil # Import shutil for directory operations

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

# Define the source tar file path and the destination extraction path on Google Drive
TAR_FILE_PATH_DRIVE = '/content/drive/MyDrive/Fakeddit_Images/public_images.tar.bz2'
DESTINATION_PATH_DRIVE = '/content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/images/public_image_set'

# Ensure the destination directory exists
os.makedirs(DESTINATION_PATH_DRIVE, exist_ok=True)

# Extract the tar.bz2 file using shell command for faster extraction
try:
    print(f"Extracting {TAR_FILE_PATH_DRIVE} to {DESTINATION_PATH_DRIVE} using shell command...")
    # Construct the shell command. -xf extracts files, -C specifies the directory.
    # We need to change to the destination directory first or use -C directly in tar command
    # Using -C is generally safer and more direct.
    # Ensure the destination path is correctly quoted in case of spaces
    shell_command = f'tar -xf "{TAR_FILE_PATH_DRIVE}" -C "{DESTINATION_PATH_DRIVE}"'
    print(f"Running command: {shell_command}")
    # Execute the shell command
    result = os.system(shell_command)

    if result == 0:
        print("Extraction complete.")
    else:
        print(f"Error during extraction. Shell command returned exit code {result}.")
        # You might want to check stderr for more details in a more robust script
        # For this example, we rely on os.system's return code

except FileNotFoundError:
    print(f"Error: Tar file not found at {TAR_FILE_PATH_DRIVE}")
except Exception as e:
    print(f"An error occurred during extraction: {e}")

Mounted at /content/drive
Google Drive mounted successfully.
Extracting /content/drive/MyDrive/Fakeddit_Images/public_images.tar.bz2 to /content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/images/public_image_set using shell command...
Running command: tar -xf "/content/drive/MyDrive/Fakeddit_Images/public_images.tar.bz2" -C "/content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/images/public_image_set"
Extraction complete.


In [ ]:
from google.colab import drive
import os

# Mount Google Drive (if not already mounted)
try:
    drive.mount('/content/drive', force_remount=True) # Use force_remount=True to be sure it's mounted
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

# Define the directory path on Google Drive
DIRECTORY_PATH_DRIVE = '/content/drive/MyDrive/Fakeddit_Images/public_image_set'

# Count the number of files in the directory using a fast shell command
try:
    print(f"Counting files in {DIRECTORY_PATH_DRIVE}...")
    # Use ls -f (fast, no sorting) and wc -l (word count lines)
    # We use -f with ls for speed as sorting is not needed for counting.
    # Ensure the path is correctly quoted
    shell_command = f'ls -f "{DIRECTORY_PATH_DRIVE}" | wc -l'
    print(f"Running command: {shell_command}")
    # Execute the shell command and capture the output
    result = os.popen(shell_command).read().strip()

    if result.isdigit():
        file_count = int(result)
        # Subtract 1 if ls includes '.' for the current directory
        # However, ls -f typically does not include '.' by default, but it's safer to check
        # For typical file listings without hidden files, this count should be accurate
        # A more robust check might involve listing and then counting in Python, but that's slower.
        # Let's assume wc -l on ls -f output gives the file count directly.
        print(f"Number of files found: {file_count}")
    else:
        print(f"Could not count files. Shell command output: {result}")

except Exception as e:
    print(f"An error occurred while counting files: {e}")

Mounted at /content/drive
Google Drive mounted successfully.
Counting files in /content/drive/MyDrive/Fakeddit_Images/public_image_set...
Running command: ls -f "/content/drive/MyDrive/Fakeddit_Images/public_image_set" | wc -l
Number of files found: 0


In [ ]:
import os
import shutil
from google.colab import drive
import pandas as pd

# Mount Google Drive (if not already mounted)
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

# Define source and base destination paths
SOURCE_DIR = '/content/drive/MyDrive/Fakeddit_Images/public_image_set'
BASE_DEST_DIR = '/content/drive/MyDrive/Fakeddit_Images/public_image_set_subfolders' # Destination subfolders will be created here

# Define the path to the training data TSV file
TRAIN_DATA_PATH = '/content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/multimodal_train.tsv.csv'

# Define target files per folder
FILES_PER_FOLDER = 9999

# Initialize folder index and file counter
folder_index = 1
file_count_in_current_folder = 0

print("Loading training data to get list of image files...")

# Load the training data to get the list of image file IDs
try:
    # Assuming the TSV file has an 'id' column
    train_df = pd.read_csv(TRAIN_DATA_PATH, sep='\t', engine='python', on_bad_lines='skip')
    # Create a list of expected image filenames from the 'id' column
    all_files_to_move = [f"{item_id}.jpg" for item_id in train_df['id'].tolist()]
    print(f"Found {len(all_files_to_move)} image files to process based on the training data.")
except FileNotFoundError:
    print(f"Error: Training data file not found at {TRAIN_DATA_PATH}. Cannot get file list.")
    all_files_to_move = [] # Proceed with empty list if file not found
except Exception as e:
    print(f"Error loading training data or processing IDs: {e}")
    all_files_to_move = [] # Proceed with empty list if error occurs


if not all_files_to_move:
    print("No image files to move based on the training data. Exiting.")
else:
    print(f"Starting file distribution from {SOURCE_DIR} to subfolders in {BASE_DEST_DIR}...")
    # Iterate through the files and move them
    for filename in all_files_to_move:
        source_path = os.path.join(SOURCE_DIR, filename)

        # Check if the source file exists before attempting to move
        if not os.path.exists(source_path):
            # print(f"Warning: Source file not found at {source_path}. Skipping.") # Optional: print warning for each missing file
            continue # Skip to the next file if the source doesn't exist

        # Determine the current destination folder path
        current_dest_folder = os.path.join(BASE_DEST_DIR, f'public_image_set_{folder_index}')

        # Create the destination folder if it doesn't exist
        os.makedirs(current_dest_folder, exist_ok=True)

        destination_path = os.path.join(current_dest_folder, filename)

        try:
            # Move the file
            shutil.move(source_path, destination_path)
            file_count_in_current_folder += 1
            # print(f"Moved {filename} to {current_dest_folder}. Count: {file_count_in_current_folder}") # Optional: print each move

            # Check if the current folder has reached the target count
            if file_count_in_current_folder >= FILES_PER_FOLDER:
                print(f"Folder {current_dest_folder} reached {FILES_PER_FOLDER} files. Moving to the next folder.")
                folder_index += 1
                file_count_in_current_folder = 0 # Reset count for the new folder

        except Exception as e:
            print(f"Error moving file {filename}: {e}")
            # Continue to the next file even if one fails

    print("\nFile distribution complete.")
    print(f"Created {folder_index} destination folders.")
    if file_count_in_current_folder > 0:
        print(f"The last folder ({os.path.join(BASE_DEST_DIR, f'public_image_set_{folder_index}')}) contains {file_count_in_current_folder} files.")

Mounted at /content/drive
Google Drive mounted successfully.
Loading training data to get list of image files...
Found 564000 image files to process based on the training data.
Starting file distribution from /content/drive/MyDrive/Fakeddit_Images/public_image_set to subfolders in /content/drive/MyDrive/Fakeddit_Images/public_image_set_subfolders...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# This script performs a full machine learning workflow for fake news detection
# using the Fakeddit dataset, multimodal (text and image) embeddings, and a neural network classifier.
# All data and models are saved to Google Drive for persistence across Colab sessions.

# --- Library Installations ---
# It's good practice to include all necessary library installations at the top
# to ensure the environment is set up correctly after a session restart.
!pip install -q chromadb sentence-transformers pandas Pillow requests

import pandas as pd
import numpy as np
import os
import chromadb
from google.colab import drive
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_score, accuracy_score, recall_score, f1_score
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from PIL import Image, ImageFile # Import ImageFile
import requests
from io import BytesIO
import tarfile
import shutil # Import shutil for file operations
import zipfile # Import zipfile for extracting zip files


# Increase the pixel limit for Pillow to handle large images
# Setting to None should disable the limit, but we'll also resize explicitly
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_PIXELS = None


print("Starting the Fakeddit Multimodal Classifier workflow...")

# 1. Mount Google Drive for persistence
try:
    drive.mount('/content/drive', force_remount=True) # Added force_remount=True
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    print("Continuing without mounting, files will not be saved persistently.")

# --- Configuration ---
# File paths for the TSV datasets and persistent storage on Google Drive
TSV_PATH_PREFIX = '/content/drive/MyDrive/FakedditDataSet/reduced_multimodal_dataset_updated' # Updated path to the new directory
TRAIN_DATA_PATH = os.path.join(TSV_PATH_PREFIX, 'multimodal_train_reduced_updated.tsv') # Using updated filename
TEST_DATA_PATH = os.path.join(TSV_PATH_PREFIX, 'multimodal_test_reduced_updated.tsv') # Using updated filename
VALIDATE_DATA_PATH = os.path.join(TSV_PATH_PREFIX, 'multimodal_validate_reduced_updated.tsv') # Using updated filename

# Local paths for ChromaDB and model
CHROMA_DB_PATH_LOCAL = '/fakedditmultimodal_embedding_subset' # Modified for subset
MODEL_SAVE_PATH_LOCAL = '/fakedditmultimodal_model_subset/model.h5' # Modified for subset

# Google Drive paths for saving
CHROMA_DB_PATH_DRIVE = '/content/drive/MyDrive/FakedditMultiModalEmbedding_Updated' # Updated path for saving embeddings
MODEL_SAVE_PATH_DRIVE = '/content/drive/MyDrive/FakedditMultiModalModel_Updated/model.h5' # Updated path for saving model

CHROMA_COLLECTION_NAME = 'fakeddit_train_multimodal_embeddings_subset' # Modified for subset


# Image paths
# LOCAL_IMAGE_EXTRACTION_PATH = '/images/public_image_set/public_image_set' # Original path
LOCAL_IMAGE_PROCESSING_PATH = '/images/images' # Updated local path for extracted images
# SOURCE_IMAGE_PATH_DRIVE = '/content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/images/public_image_set' # Source path on Drive
# TAR_FILE_PATH_DRIVE = '/content/drive/MyDrive/Fakeddit_Images/public_images.tar.bz2' # Path to the tar file
IMAGE_ZIP_PATH_DRIVE = '/content/drive/MyDrive/FakedditDataSet/reduced_multimodal_dataset/images.zip' # Path to the zip file


# Standard image size for resizing for embedding model
STANDARD_IMAGE_SIZE_EMBEDDING = (224, 224) # CLIP models often use 224x224

# Pixel limit for initial resizing to avoid DecompressionBombError
PIXEL_LIMIT = 178956970 # Keep original limit as a reference, but disabled it with Image.MAX_PIXELS = None


# --- Step 1: Data Loading ---
def load_data(file_path, limit=None):
    """Loads a .tsv file into a Pandas DataFrame and cleans it."""
    try:
        # Using engine='python' and on_bad_lines='skip' for robustness with TSV files
        df = pd.read_csv(file_path, sep='\t', engine='python', on_bad_lines='skip')
        print(f"Successfully loaded data from {file_path}. Shape: {df.shape}")

        # Drop the image_url column as it's no longer needed in the DataFrame
        if 'image_url' in df.columns:
            df.drop('image_url', axis=1, inplace=True)
            print("Dropped 'image_url' column from the DataFrame.")

        # Filter out rows where 'clean_title' is not a string or is NaN, and 'Image_URL' is missing
        original_rows = len(df)
        df = df[df['clean_title'].apply(lambda x: isinstance(x, str))].copy()
        df.dropna(subset=['id'], inplace=True) # Ensure 'id' is not missing, 'image_url' is already dropped
        print(f"Filtered out {original_rows - len(df)} records with invalid data. New shape: {df.shape}")

        if limit is not None:
            df = df.head(limit).copy() # Limit the DataFrame to the specified number of rows
            print(f"Limited data to the first {limit} records. New shape: {df.shape}")


        return df
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame(columns=['id', 'clean_title', 'title', '2_way_label']) # Return empty DataFrame on error, excluding 'image_url'
    except Exception as e:
        print(f"An error occurred while loading data from {file_path}: {e}")
        return pd.DataFrame(columns=['id', 'clean_title', 'title', '2_way_label']) # Return empty DataFrame on error, excluding 'image_url'


# Load the reduced training data
train_df = load_data(TRAIN_DATA_PATH)
# Load the reduced test and validate data for evaluation
test_df = load_data(TEST_DATA_PATH)
validate_df = load_data(VALIDATE_DATA_PATH)


if train_df.empty:
     print("Could not load training dataset. Exiting.")


# --- Extract Images Locally from Zip ---
print(f"\nExtracting images from zip file {IMAGE_ZIP_PATH_DRIVE} to {LOCAL_IMAGE_PROCESSING_PATH}...")
os.makedirs(LOCAL_IMAGE_PROCESSING_PATH, exist_ok=True)

try:
    with zipfile.ZipFile(IMAGE_ZIP_PATH_DRIVE, 'r') as zip_ref:
        # Ensure extraction is to the correct path within the processing path
        # zip_ref.extractall(LOCAL_IMAGE_PROCESSING_PATH) # This extracts *into* the path
        # If the zip contains a top-level folder named 'images', extracting to LOCAL_IMAGE_PROCESSING_PATH
        # will result in /images/images/images. We want the content of the zip's top-level 'images'
        # to be directly in /images/images. We need to extract to the parent directory and then move,
        # or carefully extract each file. The simplest is to extract to the parent and then process
        # from the expected path.
        # Based on the user's comment "images are extracted to /images/images/", it implies the
        # zip file itself might contain a directory structure like 'images/...'.
        # So, extracting to '/images' might result in '/images/images/...'.
        # Let's extract to '/images' and set LOCAL_IMAGE_PROCESSING_PATH to '/images/images'.

        extract_base_path = '/images' # Extract to the directory *containing* the final path
        os.makedirs(extract_base_path, exist_ok=True)
        zip_ref.extractall(extract_base_path)
        print(f"Image extraction complete to {extract_base_path}.")
        # Now LOCAL_IMAGE_PROCESSING_PATH points to the expected final location /images/images

except FileNotFoundError:
    print(f"Error: Image zip file not found at {IMAGE_ZIP_PATH_DRIVE}. Skipping image extraction.")
except Exception as e:
    print(f"An error occurred during zip file extraction: {e}")


# --- Step 2: Generate and Save Multimodal Embeddings to ChromaDB ---
print("\nProcessing multimodal embeddings and saving to ChromaDB...")

# Initialize Sentence-Transformers CLIP model for multimodal embeddings
embedding_model = SentenceTransformer('clip-ViT-B-32')
print("Sentence-Transformers CLIP model 'clip-ViT-B-32' loaded.")

# Get image embedding dimension
try:
    # Use the correct method to get image embedding dimension
    image_embedding_dimension = embedding_model.get_image_embedding_dimension()
    print(f"Image embedding dimension: {image_embedding_dimension}")
except AttributeError:
    # Fallback for older versions or different model structures
    print("Warning: Could not get image embedding dimension using get_image_embedding_dimension(). Attempting alternative.")
    # A common fallback is to encode a dummy image and get the dimension
    try:
        dummy_image = Image.new('RGB', (STANDARD_IMAGE_SIZE_EMBEDDING[0], STANDARD_IMAGE_SIZE_EMBEDDING[1]))
        image_embedding_dimension = embedding_model.encode(dummy_image).shape[0]
        print(f"Using dummy image encoding to get dimension: {image_embedding_dimension}")
    except Exception as e:
        print(f"Error getting image embedding dimension even with dummy image: {e}. Using a default value or you may encounter errors later.")
        image_embedding_dimension = 512 # A common dimension for CLIP models


# Initialize a persistent ChromaDB client
try:
    # Ensure the local ChromaDB directory exists before initializing the client
    os.makedirs(CHROMA_DB_PATH_LOCAL, exist_ok=True)
    print(f"Ensured ChromaDB directory exists at {CHROMA_DB_PATH_LOCAL}.")

    # Use the modified local path for the subset database
    client = chromadb.PersistentClient(path=CHROMA_DB_PATH_LOCAL)
    # Delete the collection if it already exists to regenerate embeddings
    try:
        client.delete_collection(name=CHROMA_COLLECTION_NAME)
        print(f"Deleted existing ChromaDB collection '{CHROMA_COLLECTION_NAME}'.")
    except:
        # Ignore if the collection does not exist
        pass

    collection = client.get_or_create_collection(name=CHROMA_COLLECTION_NAME)
    print(f"ChromaDB client initialized. Collection '{CHROMA_COLLECTION_NAME}' ready at {CHROMA_DB_PATH_LOCAL}.")

    # Check how many documents are already in the collection
    existing_count = collection.count()
    print(f"Found {existing_count} existing documents in the collection.")

    # Get the IDs of the documents already in the database
    existing_ids = set()
    if existing_count > 0:
        existing_ids = set(collection.get(include=[])['ids'])

    # Filter out documents that are already in the database to resume saving
    # Process only the subset of the training data loaded
    train_df_to_process = train_df[~train_df['id'].astype(str).isin(existing_ids)].copy()
    total_to_process = len(train_df_to_process)

    if total_to_process > 0:
        # Batch size for processing
        batch_size = 1000 # Adjust batch size as needed
        for i in range(0, total_to_process, batch_size):
            batch_df = train_df_to_process.iloc[i:i + batch_size]
            batch_number = i // batch_size + 1
            print(f"\nProcessing batch {batch_number} of {total_to_process // batch_size + 1}...")

            # Prepare data lists for ChromaDB's add method
            ids = []
            documents = []
            metadatas = []
            text_embeddings_batch = []
            image_embeddings_batch = []

            for index, row in batch_df.iterrows():
                item_id = str(row['id'])
                image_filename = f"{item_id}.jpg"
                # Use the local path where the subset images were extracted
                image_path = os.path.join(LOCAL_IMAGE_PROCESSING_PATH, image_filename)

                try:
                    img = Image.open(image_path)
                    img = img.resize(STANDARD_IMAGE_SIZE_EMBEDDING) # Resize the image for embedding
                    # If image is successfully loaded and resized, add to lists
                    ids.append(item_id)
                    documents.append(row['clean_title'])
                    # Exclude 'image_url' from metadata
                    metadatas.append({'title': row['title'], '2_way_label': row['2_way_label']})
                    text_embeddings_batch.append(embedding_model.encode(row['clean_title']))
                    image_embeddings_batch.append(embedding_model.encode(img))
                    # Print success message for image embedding
                    # print(f"Successfully generated image embedding for {image_filename}.")


                except (FileNotFoundError, IOError, ValueError) as e:
                    print(f"Skipping record {item_id} due to error processing local image {image_filename}: {e}")
                    # Skip this record if image processing fails
                    pass


            if ids: # Only add if there are valid records in the batch
                # Generate embeddings for the text
                # text_embeddings = embedding_model.encode(documents) # Moved inside the loop to process only included documents

                # Combine text and image embeddings by concatenation
                multimodal_embeddings = np.hstack((np.array(text_embeddings_batch), np.array(image_embeddings_batch))).tolist()

                # Add new embeddings to the collection
                collection.add(
                    ids=ids,
                    embeddings=multimodal_embeddings,
                    documents=documents,
                    metadatas=metadatas
                )
                print(f"Successfully added {len(ids)} new multimodal documents to ChromaDB.")
            else:
                print(f"No valid records with images found in batch {batch_number}. Skipping ChromaDB addition for this batch.")

    else:
        print("All documents from the training dataset subset are already in ChromaDB. Skipping embedding generation.")
except Exception as e:
    print(f"Error while generating or saving embeddings to ChromaDB: {e}")
    # In a real scenario, you might want to stop the execution here.

# --- Step 3: Train a Neural Network Model ---
print("\nTraining the Neural Network Model...")

try:
    # Check if the model is already saved locally
    if os.path.exists(MODEL_SAVE_PATH_LOCAL):
        print("Model already exists locally. Skipping training.")
        model = load_model(MODEL_SAVE_PATH_LOCAL)
    else:
        # Retrieve all IDs from the subset collection to process for training
        all_ids = collection.get(include=[])['ids']
        total_ids = len(all_ids)

        if total_ids == 0:
            print("No embeddings found in the database. Cannot train model.")
        else:
            X_train_list, y_train_list = [], []
            # Retrieve data in smaller batches to avoid "too many SQL variables" error
            batch_size = 1000 # Retrieve all 100 records in one batch
            for i in range(0, total_ids, batch_size):
                batch_ids = all_ids[i:i + batch_size]
                batch_data = collection.get(ids=batch_ids, include=['embeddings', 'metadatas'])
                X_train_list.extend(batch_data['embeddings'])
                y_train_list.extend([m['2_way_label'] for m in batch_data['metadatas']])

            X_train = np.array(X_train_list)
            y_train = np.array(y_train_list)

            # Define the Keras model
            input_dim = X_train.shape[1]
            model = Sequential([
                Dense(128, activation='relu', input_dim=input_dim),
                Dense(64, activation='relu'),
                Dense(1, activation='sigmoid')  # Sigmoid for binary classification
            ])

            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            print("Model compiled. Training...")

            # Train the model (epochs and batch size can be adjusted for the subset)
            model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, verbose=1) # Increased epochs, adjusted batch size

            # Save the trained model to local storage
            os.makedirs(os.path.dirname(MODEL_SAVE_PATH_LOCAL), exist_ok=True)
            model.save(MODEL_SAVE_PATH_LOCAL)
            print(f"Model saved successfully to: {MODEL_SAVE_PATH_LOCAL}")
except Exception as e:
    print(f"Error during model training or saving: {e}")

# --- Step 4: Model Evaluation ---
print("\n--- Model Evaluation ---")

# Load the trained model from local storage
try:
    trained_model = load_model(MODEL_SAVE_PATH_LOCAL)
    print("Trained model loaded for evaluation.")
except Exception as e:
    print(f"Error loading the trained model: {e}")
    trained_model = None

def evaluate_model(df, dataset_name):
    """
    Generates embeddings for a given DataFrame,
    makes predictions, and calculates classification metrics.
    """
    if df.empty or trained_model is None:
        print(f"Cannot evaluate on {dataset_name}: data is empty or model not loaded.")
        return

    print(f"\nEvaluating on {dataset_name} dataset...")
    # Generate multimodal embeddings for the evaluation dataset
    ids = []
    documents = []
    image_embeddings_batch = []
    y_true_batch = []

    # Use the local image processing path for evaluation images
    EVAL_IMAGE_SOURCE_PATH = LOCAL_IMAGE_PROCESSING_PATH

    for index, row in df.iterrows():
        item_id = str(row['id'])
        image_filename = f"{item_id}.jpg"
        # Use the source image path on Drive for evaluation images
        image_path = os.path.join(EVAL_IMAGE_SOURCE_PATH, image_filename)

        try:
            img = Image.open(image_path)
            img = img.resize(STANDARD_IMAGE_SIZE_EMBEDDING) # Resize the image for embedding
            # If image is successfully loaded and resized, add to lists
            ids.append(item_id)
            documents.append(row['clean_title'])
            image_embeddings_batch.append(embedding_model.encode(img))
            y_true_batch.append(row['2_way_label'])

        except (FileNotFoundError, IOError, ValueError) as e:
            print(f"Skipping evaluation record {item_id} due to error processing image {image_filename} from {EVAL_IMAGE_SOURCE_PATH}: {e}")
            # Skip this record if image processing fails
            pass


    if ids: # Only evaluate if there are valid records with images
        text_embeddings = embedding_model.encode(documents)
        X_eval = np.hstack((text_embeddings, np.array(image_embeddings_batch)))
        y_pred_probs = trained_model.predict(X_eval)
        y_pred = (y_pred_probs > 0.5).astype(int).flatten()
        y_true = np.array(y_true_batch)

        # Calculate and print metrics
        precision = precision_score(y_true, y_pred)
        accuracy = accuracy_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        print(f"--- Metrics for {dataset_name} ---")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Score: {f1:.4f}")
    else:
        print(f"No valid records with images found for {dataset_name} evaluation. Skipping evaluation.")


# Evaluate on the Test and Validation datasets
evaluate_model(test_df, "Test")
evaluate_model(validate_df, "Validation")

# --- Save local artifacts to Google Drive ---
print("\nSaving local artifacts to Google Drive...")

# Save ChromaDB embeddings
if os.path.exists(CHROMA_DB_PATH_LOCAL):
    try:
        print(f"Copying ChromaDB from {CHROMA_DB_PATH_LOCAL} to {CHROMA_DB_PATH_DRIVE}...")
        if os.path.exists(CHROMA_DB_PATH_DRIVE):
             # If destination exists, remove it to avoid issues with copying directories
             print(f"Removing existing directory at {CHROMA_DB_PATH_DRIVE}...")
             shutil.rmtree(CHROMA_DB_PATH_DRIVE)
        shutil.copytree(CHROMA_DB_PATH_LOCAL, CHROMA_DB_PATH_DRIVE)
        print("ChromaDB saved successfully to Google Drive.")
    except Exception as e:
        print(f"Error saving ChromaDB to Google Drive: {e}")
else:
    print(f"ChromaDB directory not found at {CHROMA_DB_PATH_LOCAL}. Skipping save to Drive.")

# Save trained model
if os.path.exists(os.path.dirname(MODEL_SAVE_PATH_LOCAL)):
    try:
        print(f"Copying model from {os.path.dirname(MODEL_SAVE_PATH_LOCAL)} to {os.path.dirname(MODEL_SAVE_PATH_DRIVE)}...")
        os.makedirs(os.path.dirname(MODEL_SAVE_PATH_DRIVE), exist_ok=True)
        shutil.copyfile(MODEL_SAVE_PATH_LOCAL, MODEL_SAVE_PATH_DRIVE)
        print("Model saved successfully to Google Drive.")
    except Exception as e:
        print(f"Error saving model to Google Drive: {e}")
else:
    print(f"Model directory not found at {os.path.dirname(MODEL_SAVE_PATH_LOCAL)}. Skipping save to Drive.")


print("\nWorkflow complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 71.9 MB/s eta

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 22 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 23 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 24 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 25 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 26 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 27 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 28 of 29...
Successfully added 1000 new multimodal documents to ChromaDB.

Processing batch 29 of 29...
Successfully added 592 new multimodal documents to ChromaDB.

Training the Neural Network Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model compiled. Training...
Epoch 1/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.8499 - loss: 0.3387 - val_accuracy: 0.8989 - val_loss: 0.2415
Epoch 2/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9239 - loss: 0.1938 - val_accuracy: 0.9065 - val_loss: 0.2314
Epoch 3/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9529 - loss: 0.1258 - val_accuracy: 0.9024 - val_loss: 0.2588
Epoch 4/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9774 - loss: 0.0671 - val_accuracy: 0.8946 - val_loss: 0.3255
Epoch 5/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9907 - loss: 0.0309 - val_accuracy: 0.8930 - val_loss: 0.4037
Epoch 6/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9925 - loss: 0.0211 - val_accuracy: 0.8881 - val_loss: 0.5168
Epoch 7/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9934 - loss: 0.0180 - val_accuracy: 0.8926 - val_loss: 0.5243
Epoch 8/20
715/715 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9955 - los

Model saved successfully to: /fakedditmultimodal_model_subset/model.h5

--- Model Evaluation ---
Trained model loaded for evaluation.

Evaluating on Test dataset...
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
--- Metrics for Test ---
Accuracy: 0.8716
Precision: 0.8720
Recall: 0.7963
F1 Score: 0.8325

Evaluating on Validation dataset...
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
--- Metrics for Validation ---
Accuracy: 0.8850
Precision: 0.8877
Recall: 0.8081
F1 Score: 0.8460

Saving local artifacts to Google Drive...
Copying ChromaDB from /fakedditmultimodal_embedding_subset to /content/drive/MyDrive/FakedditMultiModalEmbedding_Updated...
ChromaDB saved successfully to Google Drive.
Copying model from /fakedditmultimodal_model_subset to /content/drive/MyDrive/FakedditMultiModalModel_Updated...
Model saved successfully to Google Drive.

Workflow complete.


In [ ]:
from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

# Define the original file paths on Google Drive
TSV_PATH_PREFIX = '/content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples'
TRAIN_DATA_PATH = os.path.join(TSV_PATH_PREFIX, 'multimodal_train.tsv.csv')
TEST_DATA_PATH = os.path.join(TSV_PATH_PREFIX, 'multimodal_test_public.tsv')
VALIDATE_DATA_PATH = os.path.join(TSV_PATH_PREFIX, 'multimodal_validate.tsv')

# Define the destination directory for the reduced files on Google Drive
REDUCED_DATA_PATH_PREFIX = '/content/drive/MyDrive/FakedditDataSet/reduced_multimodal_only_samples'
os.makedirs(REDUCED_DATA_PATH_PREFIX, exist_ok=True)

# Define the paths for the reduced files
REDUCED_TRAIN_DATA_PATH = os.path.join(REDUCED_DATA_PATH_PREFIX, 'multimodal_train_reduced.tsv')
REDUCED_TEST_DATA_PATH = os.path.join(REDUCED_DATA_PATH_PREFIX, 'multimodal_test_public_reduced.tsv')
REDUCED_VALIDATE_DATA_PATH = os.path.join(REDUCED_DATA_PATH_PREFIX, 'multimodal_validate_reduced.tsv')

# Function to load, reduce, and save a TSV file
def reduce_and_save_tsv(input_path, output_path, fraction=1/3):
    """Loads a TSV file, reduces its size by the specified fraction, and saves it."""
    try:
        print(f"\nLoading data from {input_path}...")
        # Use engine='python' and on_bad_lines='skip' for potentially malformed rows
        df = pd.read_csv(input_path, sep='\t', engine='python', on_bad_lines='skip')
        print(f"Original shape: {df.shape}")

        # Sample a fraction of the data
        if len(df) > 0:
            reduced_df = df.sample(frac=fraction, random_state=42).reset_index(drop=True) # Use a fixed random_state for reproducibility
            print(f"Reduced shape: {reduced_df.shape}")

            print(f"Saving reduced data to {output_path}...")
            # Save the reduced DataFrame back to a TSV file
            reduced_df.to_csv(output_path, sep='\t', index=False)
            print("Save complete.")
        else:
            print("DataFrame is empty, skipping reduction and save.")

    except FileNotFoundError:
        print(f"Error: File not found at {input_path}. Skipping reduction.")
    except Exception as e:
        print(f"An error occurred while processing {input_path}: {e}")

# Process each file
reduce_and_save_tsv(TRAIN_DATA_PATH, REDUCED_TRAIN_DATA_PATH, fraction=1/10)
reduce_and_save_tsv(TEST_DATA_PATH, REDUCED_TEST_DATA_PATH, fraction=1/10)
reduce_and_save_tsv(VALIDATE_DATA_PATH, REDUCED_VALIDATE_DATA_PATH, fraction=1/10)

print("\nAll specified TSV files have been processed.")

Mounted at /content/drive
Google Drive mounted successfully.

Loading data from /content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/multimodal_train.tsv.csv...
Original shape: (564000, 16)
Reduced shape: (56400, 16)
Saving reduced data to /content/drive/MyDrive/FakedditDataSet/reduced_multimodal_only_samples/multimodal_train_reduced.tsv...
Save complete.

Loading data from /content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/multimodal_test_public.tsv...
Original shape: (59319, 16)
Reduced shape: (5932, 16)
Saving reduced data to /content/drive/MyDrive/FakedditDataSet/reduced_multimodal_only_samples/multimodal_test_public_reduced.tsv...
Save complete.

Loading data from /content/drive/MyDrive/FakedditDataSet/multimodal_only_samples-20250601T164248Z-1-001/multimodal_only_samples/multimodal_validate.tsv...
Original shape: (59342, 16)
Reduced shape: (5934, 16)
Saving reduced 